1. Load the King County real estate dataset created in the "Linear Regression - exercise 1" (kc_house_data_preprocessed.csv).
2. Import all necessary methods from libraries (or modules from these libraries).
3. Divide the dataset into training and test datasets.
4. For different values of the coefficient of strength of regularization:
    Using the appropriate class, train a linear regression model with Lasso regularization.
    Verify the values of the coefficients for the variables - check which ones have been zeroed out by the model.
    Calculate the metrics R2, MAE, MSE and RMSE. For which regularization coefficient is the model best? Compare these values with the metrics the linear regression model legitimized.

Using steps 1-3:

5. Using the appropriate class, train a linear regression model with Ridge regularization. Use different values of regularization strength.
6. Calculate the R2, MAE, MSE and RMSE metrics. Compare these values with the metrics legitimized by the linear regression model without regularization and the model with Lasso regularization.

In [1]:
import pandas as pd
kc_house_preprocessed_df = pd.read_csv("data/kc_house_data_preprocessed.csv")
kc_house_preprocessed_df

,Unnamed: 0,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15
0,0,221900.0,3,1.00,1180,5650,1.0,0,0,3,7,1180,0,1955,0,98178,47.5112,-122.257,1340,5650
1,1,538000.0,3,2.25,2570,7242,2.0,0,0,3,7,2170,400,1951,1991,98125,47.7210,-122.319,1690,7639
2,2,180000.0,2,1.00,770,10000,1.0,0,0,3,6,770,0,1933,0,98028,47.7379,-122.233,2720,8062
3,3,604000.0,4,3.00,1960,5000,1.0,0,0,5,7,1050,910,1965,0,98136,47.5208,-122.393,1360,5000
4,4,510000.0,3,2.00,1680,8080,1.0,0,0,3,8,1680,0,1987,0,98074,47.6168,-122.045,1800,7503
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21608,21608,360000.0,3,2.50,1530,1131,3.0,0,0,3,8,1530,0,2009,0,98103,47.6993,-122.346,1530,1509
21609,21609,400000.0,4,2.50,2310,5813,2.0,0,0,3,8,2310,0,2014,0,98146,47.5107,-122.362,1830,7200
21610,21610,402101.0,2,0.75,1020,1350,2.0,0,0,3,7,1020,0,2009,0,98144,47.5944,-122.299,1020,2007
21611,21611,400000.0,3,2.50,1600,2388,2.0,0,0,3,8,1600,0,2004,0,98027,47.5345,-122.069,1410,1287


In [2]:
kcp_house_df = kc_house_preprocessed_df.drop(columns=["Unnamed: 0"])
kcp_house_df.head()

,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15
0,221900.0,3,1.00,1180,5650,1.0,0,0,3,7,1180,0,1955,0,98178,47.5112,-122.257,1340,5650
1,538000.0,3,2.25,2570,7242,2.0,0,0,3,7,2170,400,1951,1991,98125,47.7210,-122.319,1690,7639
2,180000.0,2,1.00,770,10000,1.0,0,0,3,6,770,0,1933,0,98028,47.7379,-122.233,2720,8062
3,604000.0,4,3.00,1960,5000,1.0,0,0,5,7,1050,910,1965,0,98136,47.5208,-122.393,1360,5000
4,510000.0,3,2.00,1680,8080,1.0,0,0,3,8,1680,0,1987,0,98074,47.6168,-122.045,1800,7503


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
import numpy as np
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score

In [4]:
X = kcp_house_df.drop("price", axis=1)
y = kcp_house_df["price"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3, random_state = 42)


In [5]:
lin_reg = LinearRegression()
lin_reg.fit(X_train, y_train)

y_pred = lin_reg.predict(X_test)

In [6]:
lin_reg.score(X_test, y_test)

0.6994627057969784

In [7]:
from sklearn.linear_model import Lasso

# Create model based on Lasso Regularization
lasso_reg = Lasso(alpha=0.01)
# Fit model on training data
lasso_reg.fit(X_train, y_train)
# Predict valuse based on test data
y_lasso_pred = lasso_reg.predict(X_test)
lasso_reg.score(X_test, y_test)

c:\Users\UrosVukmanovic\anaconda3\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.696e+14, tolerance: 1.977e+11
  model = cd_fast.enet_coordinate_descent(


0.6994626719503678

In [8]:
# Calculate MSE
mse = mean_squared_error(y_test, y_lasso_pred)
mse




43387531665.674515

In [13]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.compose import TransformedTargetRegressor


def lasso_metrics(X_train, y_train, X_test, y_test, alpha_values):
    # Store one row of evaluation results for each alpha value.
    results = []

    for alpha in alpha_values:
        # Build a pipeline so the feature scaler is fit on X_train only.
        # Lasso then regularizes the standardized feature coefficients.
        lasso_pipeline = make_pipeline(
            StandardScaler(),
            Lasso(alpha=alpha, max_iter=100_000, tol=1e-4),
        )

        # Standardize the target during fitting, then convert predictions
        # back to the original price scale automatically.
        model = TransformedTargetRegressor(
            regressor=lasso_pipeline,
            transformer=StandardScaler(),
        )

        # Fit this alpha's model using training features and actual prices.
        model.fit(X_train, y_train)

        # Predict both seen training rows and unseen test rows.
        y_train_pred = model.predict(X_train)
        y_test_pred = model.predict(X_test)

        # Retrieve the fitted Lasso step to inspect its coefficients.
        lasso = model.regressor_.named_steps["lasso"]

        # Lasso sets some coefficients exactly to zero, effectively removing features.
        zeroed_features = [
            feature
            for feature, coefficient in zip(X_train.columns, lasso.coef_)
            if coefficient == 0
        ]

        # Calculate test errors in the original price units.
        mse_test = mean_squared_error(y_test, y_test_pred)

        # Save this alpha's train/test fit scores and test-set error metrics.
        results.append({
            "alpha": alpha,
            "r2_train": r2_score(y_train, y_train_pred),
            "r2_test": r2_score(y_test, y_test_pred),
            "mae_test": mean_absolute_error(y_test, y_test_pred),
            "mse_test": mse_test,
            "rmse_test": np.sqrt(mse_test),
            "zeroed_features": zeroed_features,
        })

    # Return the collected results as a table for easy comparison.
    return pd.DataFrame(results)


# Try a range of regularization strengths, from weak to strong.
alpha_values = [0.0001, 0.001, 0.01, 0.1, 1, 10]
results = lasso_metrics(X_train, y_train, X_test, y_test, alpha_values)
results

,alpha,r2_train,r2_test,mae_test,mse_test,rmse_test,zeroed_features
0,0.0001,0.699515,0.699435,127474.084536,4.339153e+10,208306.335937,[sqft_basement]
1,0.0010,0.699491,0.699164,127364.136358,4.343058e+10,208400.059566,[sqft_basement]
2,0.0100,0.697645,0.695303,126555.096582,4.398808e+10,209733.352395,"[sqft_lot, floors, sqft_basement]"
3,0.1000,0.637433,0.626843,134526.590701,5.387143e+10,232102.188163,"[bedrooms, bathrooms, sqft_lot, floors, condit..."
4,1.0000,0.000000,-0.000461,237674.864169,1.444331e+11,380043.567810,"[bedrooms, bathrooms, sqft_living, sqft_lot, f..."
5,10.0000,0.000000,-0.000461,237674.864169,1.444331e+11,380043.567810,"[bedrooms, bathrooms, sqft_living, sqft_lot, f..."


## How the Lasso Metrics Function Works

The function fits a separate Lasso model for every value of `alpha`, then collects its scores and errors in a table. `alpha` controls the strength of the penalty: stronger regularization shrinks coefficients more, and Lasso can shrink some coefficients all the way to zero. The `zeroed_features` column lists those removed predictors.

### Why Use a Pipeline?

A pipeline chains preprocessing and modeling into one estimator. Here, `StandardScaler()` standardizes the predictor columns before Lasso fits. Lasso penalizes coefficient sizes, so features measured on very different scales could otherwise receive uneven penalties. The scaler learns its means and standard deviations from `X_train` when `model.fit(...)` runs, and reuses them for predictions. This keeps test data from influencing feature scaling.

### What Does `TransformedTargetRegressor` Do?

`TransformedTargetRegressor` wraps the pipeline and standardizes the target prices using the training targets. This puts large dollar amounts on a scale that can be easier for the coordinate-descent solver to optimize, helping with convergence. When `model.predict(...)` runs, the wrapper converts predictions back to the original price scale, so MAE, MSE, and RMSE are computed in meaningful units.

### Reading the Results

- `r2_train` shows how well each model fits the training examples; `r2_test` shows performance on held-out test examples.
- `mae_test`, `mse_test`, and `rmse_test` measure test-set error. MAE and RMSE are in price units; MSE is in squared price units.
- Compare scores together: increasing `alpha` may remove more features, but too much regularization can underfit and reduce R2.

For this exercise, the table compares each `alpha` on the test set. In a real project, choose `alpha` using cross-validation on the training data, then evaluate the chosen model on the test set once. Repeatedly choosing based on test scores makes the test set part of model selection.